# 07 — kvpress: RULER Benchmark with PrefillDecodingPress

This notebook evaluates KV cache compression on the
[RULER benchmark](https://huggingface.co/datasets/simonjegou/ruler) using
[kvpress](https://github.com/NVIDIA/kvpress) with Qwen3-8B.

We test **KeyDiffPress**-based compression applied to both **prefill** and
**decoding** phases using PrefillDecodingPress, with two decoding strategies:
- **full_replacement** — CompressionRatioDecodingPress
- **filtering** — FilteringPress

RULER consists of 13 synthetic long-context tasks (NIAH variants, variable
tracking, word extraction, QA) designed to stress-test retrieval and tracking
at controlled context lengths.

Scoring uses `calculate_metrics` from the kvpress evaluation framework
(same scoring as the [kvpress leaderboard](https://huggingface.co/spaces/nvidia/kvpress-leaderboard)).

Results are saved to `results/kvpress_ruler/` for comparison in later notebooks.

## Configuration

In [ ]:
import os
os.environ["HF_HOME"] = "/opt/app-root/src/.cache/huggingface"

MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

RULER_DATA_DIRS = ["4096", "8192"]

PRESS_CONFIGS = {
    "full_replacement": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=CompressionRatioDecodingPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
        ),
    ),
    "filtering": lambda cr: PrefillDecodingPress(
        prefilling_press=KeyDiffPress(compression_ratio=cr),
        decoding_press=FilteringPress(
            base_press=KeyDiffPress(), target_compression_ratio=cr,
            fill_padding=False,
        ),
    ),
}

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/kvpress-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import kvpress
    print(f"Using FORK kvpress from {FORK_DIR}")
else:
    import kvpress
    print(f"Using SYSTEM kvpress")

print(f"  location: {os.path.dirname(kvpress.__file__)}")

EVAL_DIR = os.path.join(FORK_DIR, "evaluation")
sys.path.insert(0, EVAL_DIR)
from benchmarks.ruler.calculate_metrics import calculate_metrics as ruler_calculate_metrics
print(f"  RULER scoring from: {EVAL_DIR}")

## 1. Load Model

In [ ]:
import torch
from transformers import pipeline
from kvpress import (
    KeyDiffPress, PrefillDecodingPress, CompressionRatioDecodingPress,
    FilteringPress,
)

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} — {vram_gb:.1f} GB VRAM")

model_kwargs = {}

try:
    import flash_attn  # noqa: F401
    model_kwargs["attn_implementation"] = "flash_attention_2"
    print("Using Flash Attention 2")
except ImportError:
    print("Flash Attention 2 not available, using default attention")

pipe = pipeline(
    "kv-press-text-generation",
    model=MODEL_NAME,
    device_map="auto",
    model_kwargs=model_kwargs,
    trust_remote_code=True,
)

print(f"\nModel loaded. GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 2. Load RULER Dataset

In [ ]:
from datasets import load_dataset

ruler_datasets = {}
for data_dir in RULER_DATA_DIRS:
    ds = load_dataset("simonjegou/ruler", data_dir=data_dir, split="test")
    ruler_datasets[data_dir] = ds
    tasks = sorted(set(ds["task"]))
    print(f"RULER {data_dir}: {len(ds)} examples, {len(tasks)} tasks")
    print(f"  Tasks: {tasks}")

## 3. Run Inference

For each (algorithm, compression_ratio, context_length) combination, run all
RULER examples through the kvpress pipeline. Predictions are collected for
scoring in the next section using the kvpress evaluation framework.

In [ ]:
import time

all_results = []

configs = [("no_press", 0.0, None)]
for press_name, press_factory in PRESS_CONFIGS.items():
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio, press_factory(ratio)))

for press_name, ratio, press in configs:
    for data_dir in RULER_DATA_DIRS:
        ds = ruler_datasets[data_dir]
        label = f"{press_name} | ratio={ratio} | ctx={data_dir}"
        print(f"\n{'='*60}")
        print(f"Running: {label} ({len(ds)} examples)")
        print(f"{'='*60}")

        torch.cuda.reset_peak_memory_stats()
        start_config = time.perf_counter()

        for i, row in enumerate(ds):
            start = time.perf_counter()

            kwargs = dict(
                question=row["question"],
                answer_prefix=row["answer_prefix"],
                max_new_tokens=row["max_new_tokens"],
            )
            if press is not None:
                kwargs["press"] = press

            output = pipe(row["context"], **kwargs)
            elapsed = time.perf_counter() - start

            all_results.append({
                "framework": "kvpress",
                "press": press_name,
                "compression_ratio": ratio,
                "context_length": int(data_dir),
                "task": row["task"],
                "answer": row["answer"],
                "predicted_answer": output["answer"],
                "elapsed_sec": round(elapsed, 3),
            })

            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(ds)}")

        config_elapsed = time.perf_counter() - start_config
        peak_mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"  Done: {config_elapsed:.0f}s, peak_mem={peak_mem:.2f}GB")

        torch.cuda.empty_cache()

print(f"\nTotal results: {len(all_results)}")

## 4. Score & Results

Score predictions using `calculate_metrics` from the kvpress evaluation
framework — same string-match scoring as the kvpress leaderboard.

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []
for (press, ratio, ctx), group in df.groupby(
    ["press", "compression_ratio", "context_length"]
):
    metrics = ruler_calculate_metrics(group)
    avg_score = sum(m["string_match"] for m in metrics.values()) / len(metrics)
    mean_time = group["elapsed_sec"].mean()
    key = f"{press}__{ratio}__{ctx}"
    all_metrics[key] = metrics
    rows.append({
        "press": press, "compression_ratio": ratio,
        "context_length": ctx, "avg_score": round(avg_score, 2),
        "mean_time": round(mean_time, 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

In [ ]:
for key, metrics in sorted(all_metrics.items()):
    print(f"\n{key}")
    for task, scores in sorted(metrics.items()):
        print(f"  {task:30s}: {scores['string_match']:.2f}")

## 5. Save Results

In [ ]:
import json

os.makedirs("results/kvpress_ruler", exist_ok=True)

predictions_path = "results/kvpress_ruler/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/kvpress_ruler/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")